<a href="https://colab.research.google.com/github/TejaswiniSoni/portfolio/blob/main/Dark_Pattern_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

install libraries

In [ ]:
!pip install pandas scikit-learn joblib

Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import os
os.listdir()

['.config', 'dark_patterns.csv', 'sample_data']

Load Dataset

Check Your Data

In [ ]:
df = pd.read_csv("dark_patterns.csv")
df.head()

,text,pattern_type,severity
0,only 2 items left in stock! order now before i...,Fake Urgency,High
1,sale ends in 00:10:00 — hurry before the price...,Fake Urgency,High
2,3 people have this in their cart right now.,Fake Urgency,Medium
3,limited time offer! only valid for the next 15...,Fake Urgency,High
4,flash sale ending soon — don't miss out!,Fake Urgency,Medium


In [ ]:
df.columns
df.shape
df.isnull().sum()

,0
text,0
pattern_type,0
severity,0


Keep needed columns

In [ ]:
df = df[['text', 'pattern_type']]
df.head()

,text,pattern_type
0,only 2 items left in stock! order now before i...,Fake Urgency
1,sale ends in 00:10:00 — hurry before the price...,Fake Urgency
2,3 people have this in their cart right now.,Fake Urgency
3,limited time offer! only valid for the next 15...,Fake Urgency
4,flash sale ending soon — don't miss out!,Fake Urgency


Convert to binary label

In [ ]:
df['label'] = 'Dark'
df = df[['text', 'label']]
df.head()

,text,label
0,only 2 items left in stock! order now before i...,Dark
1,sale ends in 00:10:00 — hurry before the price...,Dark
2,3 people have this in their cart right now.,Dark
3,limited time offer! only valid for the next 15...,Dark
4,flash sale ending soon — don't miss out!,Dark


Add Not Dark data

In [ ]:
not_dark = pd.DataFrame({
    'text': [
        "add to cart",
        "go to homepage",
        "view product details",
        "continue shopping",
        "check product info",
        "open settings",
        "read more",
        "contact us",
        "about us",
        "privacy policy",
        "terms and conditions",
        "track order",
        "wishlist",
        "order history",
        "customer support",
        "see reviews",
        "browse items",
        "view profile",
        "edit account",
        "logout",
        "go back",
        "next page",
        "previous page",
        "filter results",
        "sort by price",
        "view categories",
        "open menu",
        "search product",
        "compare items",
        "add to wishlist"
    ],
    'label': ['Not Dark'] * 30
})

df = pd.concat([df, not_dark], ignore_index=True)

Clean text

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

Check labels

In [ ]:
df['label'].value_counts()

,count
label,
Dark,117
Not Dark,30


Create x and y

In [ ]:
X = df['text']
y = df['label']

In [ ]:
df.head()

,text,label
0,only 2 items left in stock order now before it...,Dark
1,sale ends in 001000 hurry before the price goe...,Dark
2,3 people have this in their cart right now,Dark
3,limited time offer only valid for the next 15 ...,Dark
4,flash sale ending soon dont miss out,Dark


In [ ]:
df['label'].value_counts()

,count
label,
Dark,117
Not Dark,30


Import F-IDf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2),
    stop_words='english'
)

X_tfidf = vectorizer.fit_transform(X)

print(X_tfidf.shape)

(147, 759)


Split data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (117, 759)
Test: (30, 759)


Train model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

Predict

In [ ]:
y_pred = model.predict(X_test)

Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7666666666666667

Classification Report:

              precision    recall  f1-score   support

        Dark       0.79      0.96      0.87        24
    Not Dark       0.00      0.00      0.00         6

    accuracy                           0.77        30
   macro avg       0.40      0.48      0.43        30
weighted avg       0.63      0.77      0.69        30


Confusion Matrix:

[[23  1]
 [ 6  0]]


Save Model + Prediction Function

In [ ]:
import joblib

joblib.dump(model, "dark_pattern_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model saved")

Model saved


Prediction function

In [ ]:
def predict_dark_pattern(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    vector = vectorizer.transform([text])
    result = model.predict(vector)[0]

    return result

Test it

In [ ]:
print(predict_dark_pattern("only 1 item left buy now"))
print(predict_dark_pattern("go to homepage"))
print(predict_dark_pattern("limited time offer hurry up"))

Dark
Not Dark
Dark


Better Flask app.py

In [ ]:
from flask import Flask, render_template, request
import joblib
import re
import numpy as np

app = Flask(__name__)

# Load trained model and vectorizer
model = joblib.load("dark_pattern_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def detect_pattern_type(text):
    if "only" in text or "hurry" in text or "limited" in text or "sale ends" in text:
        return "Fake Urgency"
    elif "no thanks" in text or "i dont want" in text or "i don't want" in text:
        return "Confirmshaming"
    elif "hidden fee" in text or "extra charge" in text:
        return "Hidden Cost"
    else:
        return "General Dark Pattern"

def detect_severity(text, confidence, prediction):
    if prediction != "Dark":
        return "Low"

    high_words = [
        "only", "hurry", "urgent", "limited", "buy now",
        "last chance", "dont miss out", "sale ends", "order now"
    ]

    score = 0
    for word in high_words:
        if word in text:
            score += 1

    if confidence is not None and confidence >= 90:
        score += 1

    if score >= 2:
        return "High"
    return "Medium"

def find_suspicious_words(text):
    risky_words = [
        "only", "hurry", "limited", "buy now", "offer",
        "sale ends", "last chance", "urgent", "order now",
        "dont miss out", "free trial", "no thanks"
    ]
    found = [word for word in risky_words if word in text]
    return found

@app.route("/", methods=["GET", "POST"])
def home():
    prediction = None
    confidence = None
    user_input = ""
    pattern_type = None
    severity = None
    suspicious_words = []

    if request.method == "POST":
        user_input = request.form.get("text", "")

        if user_input.strip():
            cleaned = clean_text(user_input)
            vector = vectorizer.transform([cleaned])

            prediction = model.predict(vector)[0]

            if hasattr(model, "predict_proba"):
                probs = model.predict_proba(vector)[0]
                confidence = round(float(np.max(probs)) * 100, 2)

            if prediction == "Dark":
                pattern_type = detect_pattern_type(cleaned)

            severity = detect_severity(cleaned, confidence, prediction)
            suspicious_words = find_suspicious_words(cleaned)

    return render_template(
        "index.html",
        prediction=prediction,
        confidence=confidence,
        user_input=user_input,
        pattern_type=pattern_type,
        severity=severity,
        suspicious_words=suspicious_words
    )

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
model = joblib.load("dark_pattern_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

In [ ]:
model = joblib.load("dark_pattern_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

In [ ]:
import joblib

model_obj = joblib.load("dark_pattern_model.pkl")
vectorizer_obj = joblib.load("tfidf_vectorizer.pkl")

print(type(model_obj))
print(type(vectorizer_obj))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [ ]:
import joblib

joblib.dump(model, "dark_pattern_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [ ]:
print(type(model))
print(type(vectorizer))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


templates/index.html

style.css

Download Model Files from Colab

In [ ]:
from google.colab import files

files.download("dark_pattern_model.pkl")
files.download("tfidf_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>